[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/15p5ur8UN3mQQwzBb2HqwoRpU0Dqd5EXm)

# Setup Spark

https://mikestaszel.com/2018/03/07/apache-spark-on-google-colaboratory/

In [0]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://apache.osuosl.org/spark/spark-2.3.1/spark-2.3.1-bin-hadoop2.7.tgz
!tar xf spark-2.3.1-bin-hadoop2.7.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-2.3.1-bin-hadoop2.7"

import findspark
findspark.init()
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = SparkContext.getOrCreate()

sc.getConf().getAll()

[('spark.app.id', 'local-1542228522372'),
 ('spark.driver.port', '41935'),
 ('spark.rdd.compress', 'True'),
 ('spark.serializer.objectStreamReset', '100'),
 ('spark.master', 'local[*]'),
 ('spark.executor.id', 'driver'),
 ('spark.submit.deployMode', 'client'),
 ('spark.driver.host', '3beeded14a0e'),
 ('spark.ui.showConsoleProgress', 'true'),
 ('spark.app.name', 'pyspark-shell')]

# Data

We'll work with a Real Estate dataset –  https://www.dropbox.com/s/vat5vyq8r713vs9/RealEstate.csv?raw=1

In [0]:
!wget https://www.dropbox.com/s/vat5vyq8r713vs9/RealEstate.csv?raw=1 -nc -O RealEstate.csv

--2018-11-14 20:43:28--  https://www.dropbox.com/s/vat5vyq8r713vs9/RealEstate.csv?raw=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.9.1, 2620:100:601f:1::a27d:901
Connecting to www.dropbox.com (www.dropbox.com)|162.125.9.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /s/raw/vat5vyq8r713vs9/RealEstate.csv [following]
--2018-11-14 20:43:28--  https://www.dropbox.com/s/raw/vat5vyq8r713vs9/RealEstate.csv
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uce9ee9044afbe5186092324168b.dl.dropboxusercontent.com/cd/0/inline/AVnBYWKrGKJZi2C8yF1OyWjpv8hjou6P9zwZC0QcdWcypGpeRiufXBH9XzPCdJOMVfs4hDJ021z6FAhebgcMH--5y2heZk6caDVHi_lI4zB4f8rdywXT0Zl4yGNWfwQW38qQWLiBGL4AROTMyX4eJD_NYQwyKJtgrQiYJiM3EPp4NiOv4YlsE7RMLUXcZSZbjzc/file [following]
--2018-11-14 20:43:29--  https://uce9ee9044afbe5186092324168b.dl.dropboxusercontent.com/cd/0/inline/AVnBYWKrGKJZi2C8yF1OyWjpv8hjou6P9zwZC0Q

In [0]:
import pandas as pd
df = pd.read_csv("RealEstate.csv")
df.head()

,MLS,Location,Price,Bedrooms,Bathrooms,Size,Price SQ Ft,Status
0,132842,Arroyo Grande,795000.0,3,3,2371,335.30,Short Sale
1,134364,Paso Robles,399000.0,4,3,2818,141.59,Short Sale
2,135141,Paso Robles,545000.0,4,3,3032,179.75,Short Sale
3,135712,Morro Bay,909000.0,4,4,3540,256.78,Short Sale
4,136282,Santa Maria-Orcutt,109900.0,3,1,1249,87.99,Short Sale


# Spark basics

### PairRDD
Pair RDD's expose operations that enable you to act on each key of a key-value in parallel to regroup/aggregate data across the network.

### GroupByKey

The "groupByKey" operator when called on a **key, value** pair  returns a dataset of (K, Iterable<V>) pairs. Let's try to leverage the "groupByKey" operator to get a listing of all locations by status. 

In [0]:
samples = sc.textFile("RealEstate.csv")
print(samples)

RealEstate.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0


In [0]:
samples.take(5)

['MLS,Location,Price,Bedrooms,Bathrooms,Size,Price SQ Ft,Status',
 '132842,Arroyo Grande,795000.00,3,3,2371,335.30,Short Sale',
 '134364,Paso Robles,399000.00,4,3,2818,141.59,Short Sale',
 '135141,Paso Robles,545000.00,4,3,3032,179.75,Short Sale',
 '135712,Morro Bay,909000.00,4,4,3540,256.78,Short Sale']

Remove header row

In [0]:
header = samples.first()
samples = samples.filter(lambda row: row != header)

In [0]:
samples.take(5)

['132842,Arroyo Grande,795000.00,3,3,2371,335.30,Short Sale',
 '134364,Paso Robles,399000.00,4,3,2818,141.59,Short Sale',
 '135141,Paso Robles,545000.00,4,3,3032,179.75,Short Sale',
 '135712,Morro Bay,909000.00,4,4,3540,256.78,Short Sale',
 '136282,Santa Maria-Orcutt,109900.00,3,1,1249,87.99,Short Sale']

In [0]:
def location_per_sample(sample):
  return [(sample.split(",")[-1], sample)]

In [0]:
from IPython import display

def get_listings(status, count):
  listings = []
  [listings.extend(each) for each in samples.map(location_per_sample).collect()]
  pairRDD = sc.parallelize(listings)
  listings_by_status = {}
  for each in pairRDD.groupByKey().collect():
    listings_by_status[each[0]] = each[1]
  display.display(list(listings_by_status[status])[0:count])

In [0]:
get_listings("Short Sale", 10)

['132842,Arroyo Grande,795000.00,3,3,2371,335.30,Short Sale',
 '134364,Paso Robles,399000.00,4,3,2818,141.59,Short Sale',
 '135141,Paso Robles,545000.00,4,3,3032,179.75,Short Sale',
 '135712,Morro Bay,909000.00,4,4,3540,256.78,Short Sale',
 '136282,Santa Maria-Orcutt,109900.00,3,1,1249,87.99,Short Sale',
 '136431,Oceano,324900.00,3,3,1800,180.50,Short Sale',
 '137036,Santa Maria-Orcutt,192900.00,4,2,1603,120.34,Short Sale',
 '137090,Santa Maria-Orcutt,215000.00,3,2,1450,148.28,Short Sale',
 '137159,Morro Bay,999000.00,4,3,3360,297.32,Short Sale',
 '137570,Atascadero,319000.00,3,2,1323,241.12,Short Sale']

In [0]:
StatusLocationPairRdd = samples.map(lambda sample:(sample.split(",")[7], sample.split(",")[1]))
LocationByStatus = StatusLocationPairRdd.groupByKey()

for status, location in LocationByStatus.collectAsMap().items():
   print("{}: {}".format(status, list(location)))

Short Sale: ['Arroyo Grande', 'Paso Robles', 'Paso Robles', 'Morro Bay', 'Santa Maria-Orcutt', 'Oceano', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Morro Bay', 'Atascadero', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Arroyo Grande', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Paso Robles', 'Los Alamos', 'San Miguel', 'Paso Robles', 'San Luis Obispo', 'Morro Bay', 'Cayucos', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Pismo Beach', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Atascadero', 'Nipomo', 'Guadalupe', 'Santa Maria-Orcutt', 'Pismo Beach', 'Santa Maria-Orcutt', 'Morro Bay', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Nipomo', 'Los Osos', 'Arroyo Grande', 'Templeton', 'Templeton', 'Grover Beach', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Santa Maria-Orcutt', 'Cambria', 'Nipom

# SparkSQL

Create a new context

In [0]:
sc = SparkContext.getOrCreate()
sc.getConf().getAll()

[('spark.app.id', 'local-1542228522372'),
 ('spark.driver.port', '41935'),
 ('spark.rdd.compress', 'True'),
 ('spark.serializer.objectStreamReset', '100'),
 ('spark.master', 'local[*]'),
 ('spark.executor.id', 'driver'),
 ('spark.submit.deployMode', 'client'),
 ('spark.driver.host', '3beeded14a0e'),
 ('spark.ui.showConsoleProgress', 'true'),
 ('spark.app.name', 'pyspark-shell')]

In [0]:
from pyspark.sql import SQLContext
sqlContext = SQLContext(sc)

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("RealEstate.csv")
df.printSchema()

root
 |-- MLS: integer (nullable = true)
 |-- Location: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Bedrooms: integer (nullable = true)
 |-- Bathrooms: integer (nullable = true)
 |-- Size: integer (nullable = true)
 |-- Price SQ Ft: double (nullable = true)
 |-- Status: string (nullable = true)



In [0]:
df.createOrReplaceTempView("listings")

### SQL examples

In [0]:
sqlDF = spark.sql("SELECT Location, Price, Bedrooms, Bathrooms, Status FROM listings")
sqlDF.show()

+------------------+--------+--------+---------+----------+
|          Location|   Price|Bedrooms|Bathrooms|    Status|
+------------------+--------+--------+---------+----------+
|     Arroyo Grande|795000.0|       3|        3|Short Sale|
|       Paso Robles|399000.0|       4|        3|Short Sale|
|       Paso Robles|545000.0|       4|        3|Short Sale|
|         Morro Bay|909000.0|       4|        4|Short Sale|
|Santa Maria-Orcutt|109900.0|       3|        1|Short Sale|
|            Oceano|324900.0|       3|        3|Short Sale|
|Santa Maria-Orcutt|192900.0|       4|        2|Short Sale|
|Santa Maria-Orcutt|215000.0|       3|        2|Short Sale|
|         Morro Bay|999000.0|       4|        3|Short Sale|
|        Atascadero|319000.0|       3|        2|Short Sale|
|Santa Maria-Orcutt|350000.0|       3|        2|Short Sale|
|Santa Maria-Orcutt|249000.0|       3|        2|Short Sale|
|     Arroyo Grande|299000.0|       2|        2|Short Sale|
|Santa Maria-Orcutt|235900.0|       3|  

records where the location is Morro Bay

In [0]:
location = "Morro Bay"
sqlDF = spark.sql("SELECT * FROM listings WHERE Location == \"%s\"" % location)
sqlDF.show(5)

+------+---------+---------+--------+---------+----+-----------+-----------+
|   MLS| Location|    Price|Bedrooms|Bathrooms|Size|Price SQ Ft|     Status|
+------+---------+---------+--------+---------+----+-----------+-----------+
|135712|Morro Bay| 909000.0|       4|        4|3540|     256.78| Short Sale|
|137159|Morro Bay| 999000.0|       4|        3|3360|     297.32| Short Sale|
|140077|Morro Bay|1100000.0|       4|        3|4168|     263.92| Short Sale|
|142528|Morro Bay| 415000.0|       3|        3|1350|     307.41| Short Sale|
|143534|Morro Bay| 789000.0|       3|        3|2100|     375.71|Foreclosure|
+------+---------+---------+--------+---------+----+-----------+-----------+
only showing top 5 rows



records with price less than 500000

In [0]:
price = 500000
sqlDF = spark.sql("SELECT * FROM listings WHERE Price < %d" % price)
print(sqlDF.count())
sqlDF.show(5)

629
+------+------------------+--------+--------+---------+----+-----------+----------+
|   MLS|          Location|   Price|Bedrooms|Bathrooms|Size|Price SQ Ft|    Status|
+------+------------------+--------+--------+---------+----+-----------+----------+
|134364|       Paso Robles|399000.0|       4|        3|2818|     141.59|Short Sale|
|136282|Santa Maria-Orcutt|109900.0|       3|        1|1249|      87.99|Short Sale|
|136431|            Oceano|324900.0|       3|        3|1800|      180.5|Short Sale|
|137036|Santa Maria-Orcutt|192900.0|       4|        2|1603|     120.34|Short Sale|
|137090|Santa Maria-Orcutt|215000.0|       3|        2|1450|     148.28|Short Sale|
+------+------------------+--------+--------+---------+----+-----------+----------+
only showing top 5 rows



records by price in descending order

In [0]:
price = 500000
sqlDF = spark.sql("SELECT * FROM listings WHERE Price < %d ORDER BY Price" % price)
print(sqlDF.count())
sqlDF.show(5)

629
+------+-------------------+-------+--------+---------+----+-----------+-----------+
|   MLS|           Location|  Price|Bedrooms|Bathrooms|Size|Price SQ Ft|     Status|
+------+-------------------+-------+--------+---------+----+-----------+-----------+
|154462| Santa Maria-Orcutt|26500.0|       2|        2|1344|      19.72|    Regular|
|148168| Santa Maria-Orcutt|29000.0|       2|        2|1500|      19.33|Foreclosure|
|154386| Santa Maria-Orcutt|36000.0|       2|        2|1056|      34.09|    Regular|
|154233|         New Cuyama|40900.0|       3|        1|1201|      34.05| Short Sale|
|154384|      Arroyo Grande|54500.0|       2|        1| 624|      87.34|    Regular|
+------+-------------------+-------+--------+---------+----+-----------+-----------+
only showing top 5 rows



count by location

In [0]:
df.groupBy("Location").count().show(5)

+-----------+-----+
|   Location|count|
+-----------+-----+
|Pismo Beach|   12|
|  King City|    3|
| New Cuyama|    1|
|     Nipomo|    3|
|     Oceano|   10|
+-----------+-----+
only showing top 5 rows



group by location and aggregate by average price

In [0]:
grouped_agg = df.groupBy("Location").avg("Price")
grouped_agg.show(5)

+-----------+-----------------+
|   Location|       avg(Price)|
+-----------+-----------------+
|Pismo Beach|772374.5833333334|
|  King City|         131190.0|
| New Cuyama|          40900.0|
|     Nipomo|454166.6666666667|
|     Oceano|         392640.0|
+-----------+-----------------+
only showing top 5 rows



group by location, aggregate the average price and sort by average price

In [0]:
grouped_agg.orderBy(grouped_agg['avg(Price)']).show(5)

+---------------+----------+
|       Location|avg(Price)|
+---------------+----------+
|     New Cuyama|   40900.0|
|Santa Margarita|   59900.0|
|    Bakersfield|   91500.0|
|      Guadalupe|  117250.0|
|      King City|  131190.0|
+---------------+----------+
only showing top 5 rows



# MLLib

In [0]:
samples = sc.textFile("RealEstate.csv")
samples.take(5)

['MLS,Location,Price,Bedrooms,Bathrooms,Size,Price SQ Ft,Status',
 '132842,Arroyo Grande,795000.00,3,3,2371,335.30,Short Sale',
 '134364,Paso Robles,399000.00,4,3,2818,141.59,Short Sale',
 '135141,Paso Robles,545000.00,4,3,3032,179.75,Short Sale',
 '135712,Morro Bay,909000.00,4,4,3540,256.78,Short Sale']

In [0]:
header = samples.first()
samples = samples.filter(lambda row: row != header)
samples.take(5)

['132842,Arroyo Grande,795000.00,3,3,2371,335.30,Short Sale',
 '134364,Paso Robles,399000.00,4,3,2818,141.59,Short Sale',
 '135141,Paso Robles,545000.00,4,3,3032,179.75,Short Sale',
 '135712,Morro Bay,909000.00,4,4,3540,256.78,Short Sale',
 '136282,Santa Maria-Orcutt,109900.00,3,1,1249,87.99,Short Sale']

Create a LabeledPoint with `Price` as the target label, and `Size`, `Bedrooms`, `Bathrooms` as the features

https://spark.apache.org/docs/2.3.0/mllib-data-types.html

In [0]:
from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.linalg import Vectors

def labeled_point_per_sample(sample):
  parts = sample.split(",")
  return LabeledPoint(parts[2], Vectors.dense(parts[3], parts[4], parts[5]))

In [0]:
labeled_points = samples.map(labeled_point_per_sample)
labeled_points.take(5)

[LabeledPoint(795000.0, [3.0,3.0,2371.0]),
 LabeledPoint(399000.0, [4.0,3.0,2818.0]),
 LabeledPoint(545000.0, [4.0,3.0,3032.0]),
 LabeledPoint(909000.0, [4.0,4.0,3540.0]),
 LabeledPoint(109900.0, [3.0,1.0,1249.0])]

In [0]:
from pyspark.mllib.regression import LinearRegressionWithSGD

labeled_points.cache()
model = LinearRegressionWithSGD.train(labeled_points, iterations=100, step=0.0000006)

Evaluate the model

In [0]:
valuesAndPreds = labeled_points.map(lambda p: (p.label, model.predict(p.features)))

In [0]:
print('   Actual  ,   Predicted')
valuesAndPreds.take(5)

   Actual  ,   Predicted


[(795000.0, 545291.5688633078),
 (399000.0, 648094.25912427),
 (545000.0, 697310.6737596794),
 (909000.0, 814142.4142878765),
 (109900.0, 287249.8637351688)]

With this setup, one can now follow all the usual steps that go in building a good Machine Learning model – Feature Engineering, Hyperparameter Tuning etc etc